In [7]:
%%writefile functions.py
import pandas as pd
import numpy as np

# Subject columns definimg
SUBJECT_COLUMNS = ['Maths', 'Physics', 'Computer_Science']

def calculate_metrics(df):
    marks_matrix = df[SUBJECT_COLUMNS].to_numpy()
    
    # NumPy array operations (row-wise calculation)
    df['Total_Marks'] = np.sum(marks_matrix, axis=1)
    df['Average_Marks'] = np.round(np.mean(marks_matrix, axis=1), 2)
    return df

def assign_grade(average):
    if average >= 90:
        return 'A+'
    elif average >= 80:
        return 'A'
    elif average >= 70:
        return 'B'
    elif average >= 60:
        return 'C'
    elif average >= 50:
        return 'D'
    else:
        return 'F'

def determine_status(row):
    subject_pass = all(row[sub] >= 40 for sub in SUBJECT_COLUMNS)
    attendance_pass = row['Attendance'] >= 75
    return 'Pass' if (subject_pass and attendance_pass) else 'Fail'

def process_student_records(df):
    grades = []
    statuses = []
    
    for _, row in df.iterrows():
        grades.append(assign_grade(row['Average_Marks']))
        statuses.append(determine_status(row))
        
    df['Grade'] = grades
    df['Status'] = statuses
    return df

def get_subject_analysis(df):
    analysis = []
    for sub in SUBJECT_COLUMNS:
        scores = df[sub].to_numpy()
        analysis.append({
            'Subject': sub,
            'Average': np.round(np.mean(scores), 2),
            'Highest': np.max(scores),
            'Lowest': np.min(scores),
            'Standard_Dev': np.round(np.std(scores), 2)
        })
    return pd.DataFrame(analysis)

Overwriting functions.py


In [9]:
%%writefile main.py
import pandas as pd
import numpy as np
from functions import calculate_metrics, process_student_records, get_subject_analysis

def main():
    print("=" * 65)
    print("       STUDENT PERFORMANCE ANALYTICS SYSTEM")
    print("=" * 65)

    # 1. Dataset Read karna (Step 2)
    try:
        df = pd.read_csv('students.csv')
        print(f"[✓] Dataset successfully load hua: {len(df)} records mile.\n")
    except FileNotFoundError:
        print("[!] Error: students.csv file nahi mili!")
        return

    # 2. Data Processing (functions.py ka use karke)
    df = calculate_metrics(df)
    df = process_student_records(df)

    # Processed table display karna
    display_cols = ['StudentID', 'Name', 'Department', 'Total_Marks', 'Average_Marks', 'Grade', 'Status']
    print("--- Processed Student Records ---")
    print(df[display_cols].to_string(index=False))
    print("\n" + "=" * 65)

    # 3. Overall Class Performance (NumPy operations)
    overall_avg = np.round(np.mean(df['Average_Marks'].to_numpy()), 2)
    highest_score = np.max(df['Average_Marks'].to_numpy())
    lowest_score = np.min(df['Average_Marks'].to_numpy())

    total_students = len(df)
    passed_count = int(np.sum(df['Status'].to_numpy() == 'Pass'))
    failed_count = int(np.sum(df['Status'].to_numpy() == 'Fail'))
    pass_pct = np.round((passed_count / total_students) * 100, 2)

    print("--- Class Performance Summary ---")
    print(f"Total Students      : {total_students}")
    print(f"Overall Class Mean  : {overall_avg}%")
    print(f"Highest Average     : {highest_score}%")
    print(f"Lowest Average      : {lowest_score}%")
    print(f"Passed Students     : {passed_count} ({pass_pct}%)")
    print(f"Failed Students     : {failed_count} ({np.round(100 - pass_pct, 2)}%)")
    print("\n" + "=" * 65)

    # 4. Subject-Wise Analysis
    print("--- Subject-Wise Performance Analysis ---")
    subject_df = get_subject_analysis(df)
    print(subject_df.to_string(index=False))
    print("\n" + "=" * 65)

    # 5. Top Performers (Top 3 Students)
    top_performers = df.sort_values(by='Average_Marks', ascending=False).head(3)
    print("--- Top 3 Students ---")
    for rank, (_, row) in enumerate(top_performers.iterrows(), start=1):
        print(f"Rank {rank}: {row['Name']} ({row['Department']}) - Avg: {row['Average_Marks']}% | Grade: {row['Grade']}")
    print("=" * 65)

if _name_ == '_main_':
    main()

Overwriting main.py
